# Vesuvius V10 - 7-Day Reliable LB Lift Plan

**Target: 0.60+ LB | Stretch: 0.62-0.66**

## Key Fixes from Previous Versions

| Issue | Fix |
|-------|-----|
| Train/val leakage | Strict volume-level 3-fold CV using scroll_id |
| Validation metric mismatch | LB-aligned full-volume eval (not patch Dice) |
| Topology schedule bug | Schedule relative to per-fold epochs (300) |
| Unstable training | AdamW + cosine decay + grad clip 8.0 |

## Training Schedule (300 epochs/fold)
- **Epochs 0-45**: Dice + BCE only (learn surfaces)
- **Epochs 45-120**: + Skeleton warmup
- **Epochs 120-300**: Full topology (+ clDice warmup)

## Loss Weights
- Dice: 0.45, BCE: 0.20, Skeleton: 0.20, clDice: 0.15

---

In [ ]:
!pip install imagecodecs connected-components-3d -q

In [ ]:
# =============================================================================
# CELL 1: IMPORTS & CONFIG
# =============================================================================

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import json
import random
import warnings
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from PIL import Image, ImageSequence
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from scipy import ndimage
from skimage.morphology import skeletonize
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

@dataclass
class Config:
    # Data paths
    DATA_ROOT: Path = Path("/kaggle/input/3d-volume-training-data")
    CHECKPOINT_DIR: Path = Path("/kaggle/working/checkpoints")
    
    # ==========================================================================
    # MODEL CONFIG
    # ==========================================================================
    PATCH_SIZE: Tuple[int, int, int] = (128, 128, 128)
    FEATURES: List[int] = field(default_factory=lambda: [32, 64, 128, 256, 320, 320])
    N_BLOCKS: List[int] = field(default_factory=lambda: [1, 2, 3, 5, 5, 5])
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    
    # ==========================================================================
    # TRAINING CONFIG
    # ==========================================================================
    EPOCHS_PER_FOLD: int = 300
    BATCH_SIZE: int = 16
    NUM_WORKERS: int = 16
    PREFETCH_FACTOR: int = 4
    
    # ==========================================================================
    # OPTIMIZER (AdamW + Cosine)
    # ==========================================================================
    LEARNING_RATE: float = 1.8e-3
    WEIGHT_DECAY: float = 1e-4
    BETAS: Tuple[float, float] = (0.9, 0.95)
    ETA_MIN: float = 1e-6
    GRADIENT_CLIP: float = 8.0
    
    # ==========================================================================
    # LOSS WEIGHTS (optimized for LB)
    # ==========================================================================
    DICE_WEIGHT: float = 0.45
    BCE_WEIGHT: float = 0.20
    SKELETON_WEIGHT: float = 0.20
    CLDICE_WEIGHT: float = 0.15
    
    # ==========================================================================
    # TOPOLOGY SCHEDULE (relative to 300 epochs)
    # ==========================================================================
    SKELETON_START_EPOCH: int = 45
    SKELETON_WARMUP_EPOCHS: int = 45
    CLDICE_START_EPOCH: int = 120
    CLDICE_WARMUP_EPOCHS: int = 60
    
    # ==========================================================================
    # VALIDATION & EARLY STOPPING
    # ==========================================================================
    FAST_VAL_EVERY: int = 5          # Patch Dice for monitoring
    LB_VAL_EVERY: int = 20           # Full-volume LB-aligned eval
    EARLY_STOP_PATIENCE: int = 60    # Epochs without improvement
    SAVE_EVERY: int = 50
    
    # ==========================================================================
    # CV CONFIG
    # ==========================================================================
    N_FOLDS: int = 3
    CV_SEED: int = 42
    
    # ==========================================================================
    # DATA
    # ==========================================================================
    PATCHES_PER_VOLUME: int = 16
    FG_OVERSAMPLE_RATIO: float = 0.6
    
    # Device
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True
    SEED: int = 42
    
    def __post_init__(self):
        self.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

cfg = Config()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

set_seed(cfg.SEED)

print("="*70)
print("V10 - 7-DAY RELIABLE LB LIFT PLAN")
print("="*70)
print(f"Target: 0.60+ LB | Stretch: 0.62-0.66")
print(f"Patch: {cfg.PATCH_SIZE} | BS: {cfg.BATCH_SIZE} | Epochs/fold: {cfg.EPOCHS_PER_FOLD}")
print(f"Folds: {cfg.N_FOLDS} | Early stop patience: {cfg.EARLY_STOP_PATIENCE}")
print("="*70)
print("Loss weights: Dice=0.45, BCE=0.20, Skel=0.20, clDice=0.15")
print(f"Schedule: Skel@{cfg.SKELETON_START_EPOCH}, clDice@{cfg.CLDICE_START_EPOCH}")
print("="*70)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("="*70)

In [ ]:
# =============================================================================
# CELL 2: STRATIFIED VOLUME FOLDS (scroll_id based)
# =============================================================================

def make_stratified_volume_folds(
    csv_path: Path,
    images_dir: Path,
    labels_dir: Path,
    n_splits: int = 3,
    seed: int = 42
) -> List[Tuple[List[str], List[str]]]:
    """
    Create stratified volume-level folds using scroll_id.
    
    CRITICAL: No volume appears in both train and val within a fold.
    Stratification ensures each fold has similar scroll distribution.
    
    Returns:
        List of (train_ids, val_ids) tuples
    """
    df = pd.read_csv(csv_path)
    
    # Filter to existing volumes
    valid_mask = df['id'].apply(
        lambda x: (images_dir / f"{x}.tif").exists() and (labels_dir / f"{x}.tif").exists()
    )
    df = df[valid_mask].reset_index(drop=True)
    
    print(f"Found {len(df)} valid volumes")
    
    # Extract scroll_id for stratification
    if 'scroll_id' in df.columns:
        strat_col = df['scroll_id'].values
    else:
        # Infer scroll_id from volume id (e.g., "scroll1_vol001" -> "scroll1")
        strat_col = df['id'].apply(lambda x: x.split('_')[0] if '_' in x else 'unknown').values
    
    print(f"Scroll distribution: {pd.Series(strat_col).value_counts().to_dict()}")
    
    # Stratified K-Fold
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    splits = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(df['id'], strat_col)):
        train_ids = df.iloc[train_idx]['id'].tolist()
        val_ids = df.iloc[val_idx]['id'].tolist()
        
        # Verify no overlap
        assert len(set(train_ids) & set(val_ids)) == 0, "Train/val overlap detected!"
        
        splits.append((train_ids, val_ids))
        print(f"Fold {fold}: Train={len(train_ids)}, Val={len(val_ids)}")
    
    return splits


def build_epoch_schedule(total_epochs: int) -> Dict[str, int]:
    """
    Build topology schedule relative to total epochs.
    
    For 300 epochs:
    - skel_start=45, skel_warmup=45
    - cldice_start=120, cldice_warmup=60
    """
    ratio = total_epochs / 300
    return {
        'skel_start': int(45 * ratio),
        'skel_warmup': int(45 * ratio),
        'cldice_start': int(120 * ratio),
        'cldice_warmup': int(60 * ratio),
    }


# Create folds
train_csv = cfg.DATA_ROOT / "train.csv"
train_images = cfg.DATA_ROOT / "train_images"
train_labels = cfg.DATA_ROOT / "train_labels"

if train_csv.exists():
    FOLD_SPLITS = make_stratified_volume_folds(
        train_csv, train_images, train_labels,
        n_splits=cfg.N_FOLDS, seed=cfg.CV_SEED
    )
    
    # Save fold info for reproducibility
    fold_info = {
        'n_folds': cfg.N_FOLDS,
        'seed': cfg.CV_SEED,
        'folds': [{'train': t, 'val': v} for t, v in FOLD_SPLITS]
    }
    print(f"\nFold splits saved. Total: {sum(len(t)+len(v) for t,v in FOLD_SPLITS)//cfg.N_FOLDS} unique volumes")
else:
    print("train.csv not found - running in test mode")
    FOLD_SPLITS = []

In [ ]:
# =============================================================================
# CELL 3: MODEL (ResEncUNet3D)
# =============================================================================

class SafeInstanceNorm3d(nn.Module):
    def __init__(self, num_features, eps=1e-5, affine=True):
        super().__init__()
        self.eps = eps
        self.affine = affine
        if affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))
    
    def forward(self, x):
        mean = x.mean(dim=[2,3,4], keepdim=True)
        var = x.var(dim=[2,3,4], keepdim=True, unbiased=False)
        x = (x - mean) / torch.sqrt(var.clamp(min=self.eps) + self.eps)
        if self.affine:
            x = self.weight.view(1,-1,1,1,1) * x + self.bias.view(1,-1,1,1,1)
        return x


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            SafeInstanceNorm3d(out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(*[ConvBlock(channels, channels) for _ in range(n_convs)])
    
    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)
    
    def forward(self, x):
        b, c = x.shape[:2]
        cse = torch.sigmoid(self.cse_fc2(F.relu(self.cse_fc1(self.cse_pool(x).view(b,c))))).view(b,c,1,1,1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_scse=True, use_deep_supervision=True):
        super().__init__()
        
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 2, 3, 5, 5, 5]
        
        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        
        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i-1]
            self.encoders.append(nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat) for _ in range(nb)]
            ))
            self.attentions.append(scSEBlock(feat) if use_scse else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))
        
        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features)-2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i+1], features[i], 2, stride=2))
            self.dec_convs.append(ConvBlock(features[i]*2, features[i]))
        
        self.final = nn.Conv3d(features[0], out_ch, 1)
        
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList([
                nn.Conv3d(features[-(i+2)], out_ch, 1) 
                for i in range(min(3, len(features)-1))
            ])
    
    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        ds_outputs = []
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i+1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            if self.use_deep_supervision and self.training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        out = self.final(x)
        if self.use_deep_supervision and self.training:
            return {'output': out, 'deep': ds_outputs}
        return out


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

_m = ResEncUNet3D(features=cfg.FEATURES, n_blocks=cfg.N_BLOCKS)
print(f"Model: {count_params(_m)/1e6:.1f}M params")
del _m

In [ ]:
# =============================================================================
# CELL 4: LOSS FUNCTIONS (V10 schedule)
# =============================================================================

def soft_skeletonize(x, num_iter=10):
    for _ in range(num_iter):
        min_pool = -F.max_pool3d(-x, 3, stride=1, padding=1)
        max_min_pool = F.max_pool3d(min_pool, 3, stride=1, padding=1)
        x = F.relu(x - max_min_pool)
    return x


class CombinedLossV10(nn.Module):
    """
    V10 Loss with correct schedule for 300 epochs:
    - Epochs 0-45: Dice + BCE only
    - Epochs 45-120: + Skeleton warmup
    - Epochs 120-300: + clDice warmup
    """
    def __init__(self, dice_w=0.45, bce_w=0.20, skel_w=0.20, cldice_w=0.15,
                 skel_start=45, skel_warmup=45, cldice_start=120, cldice_warmup=60):
        super().__init__()
        self.dice_w = dice_w
        self.bce_w = bce_w
        self.skel_w = skel_w
        self.cldice_w = cldice_w
        
        self.skel_start = skel_start
        self.skel_warmup = skel_warmup
        self.cldice_start = cldice_start
        self.cldice_warmup = cldice_warmup
        
        self.ds_weights = [0.5, 0.25, 0.125]
    
    def _scale(self, epoch, start, warmup):
        if epoch < start:
            return 0.0
        elif epoch < start + warmup:
            return (epoch - start) / warmup
        return 1.0
    
    def dice_loss(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        inter = (pred * target).sum()
        union = pred.sum() + target.sum()
        return 1 - (2 * inter + 1e-5) / (union + 1e-5)
    
    def bce_loss(self, pred, target, mask=None):
        if mask is not None:
            valid = 1 - mask
            loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
            return (loss * valid).sum() / (valid.sum() + 1e-8)
        return F.binary_cross_entropy_with_logits(pred, target)
    
    def skeleton_loss(self, pred, target, mask=None):
        pred_sig = torch.sigmoid(pred)
        if mask is not None:
            pred_sig = pred_sig * (1 - mask)
            target = target * (1 - mask)
        target_skel = soft_skeletonize(target, num_iter=10)
        target_tube = F.max_pool3d(target_skel, 5, stride=1, padding=2)
        recall = (pred_sig * target_tube).sum() / (target_tube.sum() + 1e-5)
        return 1 - recall
    
    def cldice_loss(self, pred, target, mask=None):
        pred_sig = torch.sigmoid(pred)
        if mask is not None:
            pred_sig = pred_sig * (1 - mask)
            target = target * (1 - mask)
        skel_pred = soft_skeletonize(pred_sig, 10)
        skel_target = soft_skeletonize(target, 10)
        tprec = ((skel_pred * target).sum() + 1e-5) / (skel_pred.sum() + 1e-5)
        tsens = ((skel_target * pred_sig).sum() + 1e-5) / (skel_target.sum() + 1e-5)
        return 1 - 2 * tprec * tsens / (tprec + tsens + 1e-5)
    
    def forward(self, output, target, ignore_mask, epoch):
        if isinstance(output, dict):
            pred = output['output']
            deep = output.get('deep', [])
        else:
            pred = output
            deep = []
        
        skel_scale = self._scale(epoch, self.skel_start, self.skel_warmup)
        cldice_scale = self._scale(epoch, self.cldice_start, self.cldice_warmup)
        
        dice = self.dice_loss(pred, target, ignore_mask)
        bce = self.bce_loss(pred, target, ignore_mask)
        
        if skel_scale > 0:
            skel = self.skeleton_loss(pred, target, ignore_mask)
        else:
            skel = torch.tensor(0.0, device=pred.device)
        
        if cldice_scale > 0:
            cldice = self.cldice_loss(pred, target, ignore_mask)
        else:
            cldice = torch.tensor(0.0, device=pred.device)
        
        total = (self.dice_w * dice + 
                 self.bce_w * bce + 
                 self.skel_w * skel_scale * skel + 
                 self.cldice_w * cldice_scale * cldice)
        
        # Deep supervision
        for i, ds in enumerate(deep):
            if i >= len(self.ds_weights):
                break
            ds_target = F.interpolate(target, size=ds.shape[2:], mode='nearest')
            ds_mask = F.interpolate(ignore_mask, size=ds.shape[2:], mode='nearest')
            total = total + self.ds_weights[i] * self.dice_loss(ds, ds_target, ds_mask)
        
        return {
            'total': total,
            'dice': dice.item(),
            'bce': bce.item(),
            'skel': skel.item() if skel_scale > 0 else 0,
            'cldice': cldice.item() if cldice_scale > 0 else 0,
            'skel_scale': skel_scale,
            'cldice_scale': cldice_scale,
        }

print("V10 Loss schedule:")
print(f"  Epochs 0-{cfg.SKELETON_START_EPOCH}: Dice + BCE")
print(f"  Epochs {cfg.SKELETON_START_EPOCH}-{cfg.CLDICE_START_EPOCH}: + Skeleton")
print(f"  Epochs {cfg.CLDICE_START_EPOCH}+: + clDice")

In [ ]:
# =============================================================================
# CELL 5: LB-ALIGNED METRICS
# =============================================================================

try:
    import cc3d
    USE_CC3D = True
except:
    USE_CC3D = False

def connected_components_3d(vol, connectivity=26):
    if USE_CC3D:
        return cc3d.connected_components(vol.astype(np.uint8), connectivity=connectivity)
    else:
        struct = ndimage.generate_binary_structure(3, 3 if connectivity == 26 else 1)
        labeled, _ = ndimage.label(vol, structure=struct)
        return labeled


def compute_dice(pred: np.ndarray, gt: np.ndarray, ignore_mask: np.ndarray = None) -> float:
    """Compute Dice score."""
    if ignore_mask is not None:
        pred = pred.copy()
        gt = gt.copy()
        pred[ignore_mask] = 0
        gt[ignore_mask] = 0
    
    inter = (pred & gt).sum()
    union = pred.sum() + gt.sum()
    return (2 * inter + 1e-5) / (union + 1e-5)


def compute_surface_dice(pred: np.ndarray, gt: np.ndarray, tolerance: float = 2.0) -> float:
    """
    Compute Surface Dice (approximation).
    
    Measures agreement of surfaces within tolerance distance.
    """
    from scipy.ndimage import distance_transform_edt
    
    # Get surfaces (boundary voxels)
    pred_surface = pred & ~ndimage.binary_erosion(pred)
    gt_surface = gt & ~ndimage.binary_erosion(gt)
    
    if pred_surface.sum() == 0 or gt_surface.sum() == 0:
        return 0.0
    
    # Distance transforms
    pred_dist = distance_transform_edt(~pred_surface)
    gt_dist = distance_transform_edt(~gt_surface)
    
    # Surface points within tolerance
    pred_within = (gt_dist[pred_surface] <= tolerance).sum()
    gt_within = (pred_dist[gt_surface] <= tolerance).sum()
    
    precision = pred_within / pred_surface.sum()
    recall = gt_within / gt_surface.sum()
    
    return (2 * precision * recall) / (precision + recall + 1e-8)


def compute_voi(pred: np.ndarray, gt: np.ndarray, connectivity: int = 26) -> float:
    """
    Compute Variation of Information (VOI) score.
    
    Lower VOI = better segmentation.
    Returns 1 - normalized_voi for scoring (higher = better).
    """
    pred_labels = connected_components_3d(pred, connectivity)
    gt_labels = connected_components_3d(gt, connectivity)
    
    # Compute contingency table
    n = pred.size
    
    # Get unique labels
    pred_unique = np.unique(pred_labels)
    gt_unique = np.unique(gt_labels)
    
    # Entropy of pred
    pred_counts = np.array([(pred_labels == i).sum() for i in pred_unique])
    pred_probs = pred_counts / n
    h_pred = -np.sum(pred_probs * np.log2(pred_probs + 1e-10))
    
    # Entropy of gt
    gt_counts = np.array([(gt_labels == i).sum() for i in gt_unique])
    gt_probs = gt_counts / n
    h_gt = -np.sum(gt_probs * np.log2(gt_probs + 1e-10))
    
    # Mutual information (simplified)
    mi = 0
    for pi in pred_unique[:min(50, len(pred_unique))]:  # Limit for speed
        for gi in gt_unique[:min(50, len(gt_unique))]:
            joint = ((pred_labels == pi) & (gt_labels == gi)).sum()
            if joint > 0:
                p_joint = joint / n
                p_pred = (pred_labels == pi).sum() / n
                p_gt = (gt_labels == gi).sum() / n
                mi += p_joint * np.log2(p_joint / (p_pred * p_gt + 1e-10) + 1e-10)
    
    voi = h_pred + h_gt - 2 * mi
    
    # Normalize and invert (higher = better)
    max_voi = h_pred + h_gt
    if max_voi > 0:
        return 1 - (voi / max_voi)
    return 1.0


def evaluate_leaderboard(
    pred: np.ndarray,
    gt: np.ndarray,
    ignore_label: int = 2,
    surface_tolerance: float = 2.0,
    voi_connectivity: int = 26
) -> Dict[str, float]:
    """
    Compute LB-aligned metrics.
    
    LB Score = 0.30 * TopoScore + 0.35 * SurfaceDice + 0.35 * VOI
    
    Note: TopoScore requires Betti numbers (not implemented here).
    We use a surrogate based on component count similarity.
    """
    # Create binary masks
    pred_bin = (pred > 0).astype(np.uint8)
    gt_bin = (gt == 1).astype(np.uint8)
    ignore_mask = (gt == ignore_label)
    
    # Apply ignore mask
    pred_bin[ignore_mask] = 0
    gt_bin[ignore_mask] = 0
    
    # Compute metrics
    dice = compute_dice(pred_bin, gt_bin)
    surface_dice = compute_surface_dice(pred_bin, gt_bin, surface_tolerance)
    voi_score = compute_voi(pred_bin, gt_bin, voi_connectivity)
    
    # Topology surrogate (component count similarity)
    pred_cc = connected_components_3d(pred_bin).max()
    gt_cc = connected_components_3d(gt_bin).max()
    topo_surrogate = 1.0 - min(abs(pred_cc - gt_cc) / max(gt_cc, 1), 1.0)
    
    # Composite score (LB-like)
    lb_score = 0.30 * topo_surrogate + 0.35 * surface_dice + 0.35 * voi_score
    
    return {
        'dice': dice,
        'surface_dice': surface_dice,
        'voi_score': voi_score,
        'topo_surrogate': topo_surrogate,
        'lb_score': lb_score,
        'pred_components': pred_cc,
        'gt_components': gt_cc,
    }

print("LB-aligned metrics ready")
print("  LB = 0.30*Topo + 0.35*SurfaceDice + 0.35*VOI")

In [ ]:
# =============================================================================
# CELL 6: DATASET (PRELOAD ALL - 220GB RAM available)
# =============================================================================

def load_volume(path):
    try:
        import tifffile
        return tifffile.imread(str(path))
    except:
        im = Image.open(path)
        return np.stack([np.array(p) for p in ImageSequence.Iterator(im)], axis=0)


# Global cache to share across train/val datasets
VOLUME_CACHE = {}
FG_COORDS_CACHE = {}


def preload_volumes(volume_ids, images_dir, labels_dir):
    """Preload volumes into global cache (shared across datasets)."""
    global VOLUME_CACHE, FG_COORDS_CACHE
    
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)
    
    to_load = [vid for vid in volume_ids if vid not in VOLUME_CACHE]
    
    if len(to_load) == 0:
        print(f"All {len(volume_ids)} volumes already cached")
        return
    
    print(f"Preloading {len(to_load)} volumes (already cached: {len(volume_ids) - len(to_load)})...")
    
    for vid in tqdm(to_load, desc="Loading"):
        img = load_volume(images_dir / f"{vid}.tif").astype(np.float32)
        lbl = load_volume(labels_dir / f"{vid}.tif").astype(np.uint8)
        img = (img - img.mean()) / (img.std() + 1e-8)
        VOLUME_CACHE[vid] = (img, lbl)
        
        fg = np.argwhere(lbl == 1)
        if len(fg) > 10000:
            fg = fg[np.random.choice(len(fg), 10000, replace=False)]
        FG_COORDS_CACHE[vid] = fg if len(fg) > 0 else None
    
    gb = sum(v[0].nbytes + v[1].nbytes for v in VOLUME_CACHE.values()) / 1e9
    print(f"Total cached: {len(VOLUME_CACHE)} volumes ({gb:.1f} GB)")


class VesuviusDatasetV10(Dataset):
    """
    Dataset for V10 - uses global preloaded cache.
    With 220GB RAM, preload everything for max speed.
    """
    def __init__(self, volume_ids, images_dir, labels_dir, patch_size=(128,128,128),
                 is_train=True, patches_per_volume=16, fg_oversample=0.6):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        self.fg_oversample = fg_oversample
        self.volume_ids = volume_ids
        
        # Preload into global cache
        preload_volumes(volume_ids, images_dir, labels_dir)
        
        print(f"Dataset ready: {len(self)} samples")
    
    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume
    
    def __getitem__(self, idx):
        vid = self.volume_ids[idx // self.patches_per_volume]
        img, lbl = VOLUME_CACHE[vid]
        
        d, h, w = img.shape
        pd, ph, pw = self.patch_size
        
        if d < pd or h < ph or w < pw:
            img = np.pad(img, ((0,max(0,pd-d)),(0,max(0,ph-h)),(0,max(0,pw-w))), mode='reflect')
            lbl = np.pad(lbl, ((0,max(0,pd-d)),(0,max(0,ph-h)),(0,max(0,pw-w))), mode='constant', constant_values=2)
            d, h, w = img.shape
        
        fg = FG_COORDS_CACHE.get(vid)
        if self.is_train and random.random() < self.fg_oversample and fg is not None and len(fg) > 0:
            c = fg[random.randint(0, len(fg)-1)]
            z = max(0, min(c[0] - pd//2, d - pd))
            y = max(0, min(c[1] - ph//2, h - ph))
            x = max(0, min(c[2] - pw//2, w - pw))
        else:
            z = random.randint(0, max(0, d - pd))
            y = random.randint(0, max(0, h - ph))
            x = random.randint(0, max(0, w - pw))
        
        img_p = img[z:z+pd, y:y+ph, x:x+pw].copy()
        lbl_p = lbl[z:z+pd, y:y+ph, x:x+pw].copy()
        
        if self.is_train:
            for ax in range(3):
                if random.random() > 0.5:
                    img_p = np.flip(img_p, ax)
                    lbl_p = np.flip(lbl_p, ax)
            k = random.randint(0, 3)
            if k:
                img_p = np.rot90(img_p, k, (1,2))
                lbl_p = np.rot90(lbl_p, k, (1,2))
            img_p = np.ascontiguousarray(img_p)
            lbl_p = np.ascontiguousarray(lbl_p)
            if random.random() > 0.5:
                img_p = img_p * random.uniform(0.8, 1.2)
            if random.random() > 0.5:
                img_p = img_p + random.uniform(-0.1, 0.1)
        
        fg_mask = (lbl_p == 1).astype(np.float32)
        ig_mask = (lbl_p == 2).astype(np.float32)
        
        return {
            'image': torch.from_numpy(img_p).unsqueeze(0).float(),
            'label': torch.from_numpy(fg_mask).unsqueeze(0).float(),
            'ignore_mask': torch.from_numpy(ig_mask).unsqueeze(0).float(),
        }

print("Dataset ready (preload all - 220GB RAM)")

In [ ]:
# =============================================================================
# CELL 7: FULL-VOLUME INFERENCE
# =============================================================================

@torch.no_grad()
def predict_volume(model, volume, patch_size=(128,128,128), overlap=0.5, device='cuda'):
    """
    Predict full volume with overlapping patches and Gaussian weighting.
    """
    model.eval()
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd * (1-overlap)), int(ph * (1-overlap)), int(pw * (1-overlap))
    
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count = np.zeros((D, H, W), dtype=np.float32)
    
    # Gaussian weighting
    sigma = 0.125
    gz = np.exp(-0.5 * ((np.arange(pd) - pd/2) / (pd*sigma))**2)
    gy = np.exp(-0.5 * ((np.arange(ph) - ph/2) / (ph*sigma))**2)
    gx = np.exp(-0.5 * ((np.arange(pw) - pw/2) / (pw*sigma))**2)
    gauss = (gz[:,None,None] * gy[None,:,None] * gx[None,None,:]).astype(np.float32)
    
    # Positions
    z_pos = list(range(0, max(1, D-pd+1), sd))
    if D > pd and (D-pd) not in z_pos: z_pos.append(D-pd)
    y_pos = list(range(0, max(1, H-ph+1), sh))
    if H > ph and (H-ph) not in y_pos: y_pos.append(H-ph)
    x_pos = list(range(0, max(1, W-pw+1), sw))
    if W > pw and (W-pw) not in x_pos: x_pos.append(W-pw)
    
    # Normalize
    vol_norm = (volume.astype(np.float32) - volume.mean()) / (volume.std() + 1e-8)
    
    for z in z_pos:
        for y in y_pos:
            for x in x_pos:
                patch = vol_norm[z:z+pd, y:y+ph, x:x+pw].copy()
                actual = patch.shape
                
                if patch.shape != (pd, ph, pw):
                    patch = np.pad(patch, [(0,pd-patch.shape[0]), (0,ph-patch.shape[1]), (0,pw-patch.shape[2])], mode='reflect')
                
                inp = torch.from_numpy(patch[None,None].astype(np.float32)).to(device)
                
                with autocast(enabled=cfg.USE_AMP):
                    out = model(inp)
                    if isinstance(out, dict): out = out['output']
                    pred = torch.sigmoid(out).squeeze().cpu().numpy()
                
                out_crop = pred[:actual[0], :actual[1], :actual[2]]
                w_crop = gauss[:actual[0], :actual[1], :actual[2]]
                
                pred_sum[z:z+out_crop.shape[0], y:y+out_crop.shape[1], x:x+out_crop.shape[2]] += out_crop * w_crop
                count[z:z+out_crop.shape[0], y:y+out_crop.shape[1], x:x+out_crop.shape[2]] += w_crop
    
    return pred_sum / np.maximum(count, 1e-8)


@torch.no_grad()
def validate_fast(model, loader, device):
    """Fast patch-based validation (Dice only, for monitoring)."""
    model.eval()
    total_dice = 0
    n = 0
    
    for batch in loader:
        images = batch['image'].to(device)
        labels = batch['label'].numpy()
        ignore = batch['ignore_mask'].numpy()
        
        with autocast(enabled=cfg.USE_AMP):
            out = model(images)
            if isinstance(out, dict): out = out['output']
            probs = torch.sigmoid(out).cpu().numpy()
        
        for b in range(images.shape[0]):
            pred = (probs[b,0] > 0.5).astype(np.uint8)
            tgt = labels[b,0].astype(np.uint8)
            ign = ignore[b,0] > 0.5
            pred[ign] = 0
            tgt[ign] = 0
            inter = (pred & tgt).sum()
            union = pred.sum() + tgt.sum()
            total_dice += (2*inter + 1e-5) / (union + 1e-5)
            n += 1
    
    return total_dice / max(n, 1)


@torch.no_grad()
def validate_lb_aligned(model, val_ids, images_dir, labels_dir, device, max_volumes=3):
    """
    LB-aligned full-volume validation.
    
    Evaluates on full volumes with proper metrics.
    """
    model.eval()
    metrics_list = []
    
    for vid in val_ids[:max_volumes]:
        img = load_volume(images_dir / f"{vid}.tif")
        lbl = load_volume(labels_dir / f"{vid}.tif")
        
        # Predict
        prob = predict_volume(model, img, cfg.PATCH_SIZE, overlap=0.5, device=device)
        pred = (prob > 0.3).astype(np.uint8)
        
        # Evaluate
        metrics = evaluate_leaderboard(pred, lbl)
        metrics_list.append(metrics)
    
    # Average
    avg_metrics = {}
    for key in metrics_list[0].keys():
        if isinstance(metrics_list[0][key], (int, float)):
            avg_metrics[key] = np.mean([m[key] for m in metrics_list])
    
    return avg_metrics

print("Inference functions ready")

In [ ]:
# =============================================================================
# CELL 8: FOLD TRAINING (preload all - 220GB RAM)
# =============================================================================

import sys
import time

def train_fold_v10(fold: int, train_ids: List[str], val_ids: List[str]) -> Dict:
    """
    Train a single fold with V10 settings.
    Uses global preloaded cache (220GB RAM available).

    Returns:
        Dict with best_score, oof_probs, metrics history
    """
    print("="*70)
    print(f"FOLD {fold} TRAINING")
    print(f"Train: {len(train_ids)} volumes | Val: {len(val_ids)} volumes")
    print("="*70)

    # Paths
    train_images = cfg.DATA_ROOT / "train_images"
    train_labels = cfg.DATA_ROOT / "train_labels"

    # Datasets using GLOBAL CACHE (preloads everything)
    train_ds = VesuviusDatasetV10(
        train_ids, train_images, train_labels,
        patch_size=cfg.PATCH_SIZE, is_train=True,
        patches_per_volume=cfg.PATCHES_PER_VOLUME,
        fg_oversample=cfg.FG_OVERSAMPLE_RATIO,
    )
    val_ds = VesuviusDatasetV10(
        val_ids, train_images, train_labels,
        patch_size=cfg.PATCH_SIZE, is_train=False,
        patches_per_volume=4,
    )

    train_loader = DataLoader(
        train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
        num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True,
        persistent_workers=True,  # Keep workers alive for speed
        prefetch_factor=cfg.PREFETCH_FACTOR,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
        num_workers=4, pin_memory=True,
    )

    print(f"Train: {len(train_ds)} patches | Val: {len(val_ds)} patches")

    # Model
    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
    ).to(cfg.DEVICE)

    if hasattr(torch, 'compile'):
        model = torch.compile(model, mode='reduce-overhead')

    print(f"Model: {count_params(model)/1e6:.1f}M params")

    # Loss
    criterion = CombinedLossV10(
        dice_w=cfg.DICE_WEIGHT, bce_w=cfg.BCE_WEIGHT,
        skel_w=cfg.SKELETON_WEIGHT, cldice_w=cfg.CLDICE_WEIGHT,
        skel_start=cfg.SKELETON_START_EPOCH, skel_warmup=cfg.SKELETON_WARMUP_EPOCHS,
        cldice_start=cfg.CLDICE_START_EPOCH, cldice_warmup=cfg.CLDICE_WARMUP_EPOCHS,
    )

    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.LEARNING_RATE,
        weight_decay=cfg.WEIGHT_DECAY,
        betas=cfg.BETAS,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.EPOCHS_PER_FOLD, eta_min=cfg.ETA_MIN,
    )
    scaler = GradScaler(enabled=cfg.USE_AMP)

    # Checkpoint dir
    fold_dir = cfg.CHECKPOINT_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    # Training state
    best_lb_score = -1
    best_epoch = -1
    patience_counter = 0
    history = []

    for epoch in range(cfg.EPOCHS_PER_FOLD):
        t0 = time.time()
        model.train()

        total_loss = 0
        n = 0

        pbar = tqdm(train_loader, desc=f"F{fold} E{epoch+1}", file=sys.stdout, leave=False)
        for batch in pbar:
            images = batch['image'].to(cfg.DEVICE, non_blocking=True)
            labels = batch['label'].to(cfg.DEVICE, non_blocking=True)
            ignore = batch['ignore_mask'].to(cfg.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=cfg.USE_AMP):
                out = model(images)
                losses = criterion(out, labels, ignore, epoch)

            scaler.scale(losses['total']).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
            scaler.step(optimizer)
            scaler.update()

            total_loss += losses['total'].item()
            n += 1

            pbar.set_postfix(
                loss=f"{losses['total'].item():.3f}",
                skel=f"{losses['skel_scale']:.2f}",
                cld=f"{losses['cldice_scale']:.2f}",
            )

        scheduler.step()

        train_loss = total_loss / n
        dt = time.time() - t0

        # Phase
        if epoch < cfg.SKELETON_START_EPOCH:
            phase = "DICE"
        elif epoch < cfg.CLDICE_START_EPOCH:
            phase = "SKEL"
        else:
            phase = "TOPO"

        # Fast validation
        fast_dice = 0
        if (epoch + 1) % cfg.FAST_VAL_EVERY == 0:
            fast_dice = validate_fast(model, val_loader, cfg.DEVICE)

        # LB-aligned validation (less frequent to save time)
        lb_score = 0
        if (epoch + 1) % cfg.LB_VAL_EVERY == 0:
            lb_metrics = validate_lb_aligned(
                model, val_ids, train_images, train_labels, cfg.DEVICE, max_volumes=2
            )
            lb_score = lb_metrics['lb_score']

            if lb_score > best_lb_score:
                best_lb_score = lb_score
                best_epoch = epoch
                patience_counter = 0

                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_lb_score': best_lb_score,
                    'fold': fold,
                    'metrics': lb_metrics,
                }, fold_dir / 'best_model.pth')

                print(f"  >>> New best LB: {lb_score:.4f}")
            else:
                patience_counter += cfg.LB_VAL_EVERY

        # Log
        log_str = f"F{fold} E{epoch+1}/{cfg.EPOCHS_PER_FOLD} [{phase}] | {dt:.1f}s | Loss: {train_loss:.4f}"
        if fast_dice > 0:
            log_str += f" | Dice: {fast_dice:.4f}"
        if lb_score > 0:
            log_str += f" | LB: {lb_score:.4f}"
        print(log_str)

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'fast_dice': fast_dice,
            'lb_score': lb_score,
            'phase': phase,
        })

        # Early stopping
        if patience_counter >= cfg.EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

        # Periodic save
        if (epoch + 1) % cfg.SAVE_EVERY == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
            }, fold_dir / f'checkpoint_epoch_{epoch+1}.pth')

    print(f"\nFold {fold} complete. Best LB: {best_lb_score:.4f} @ epoch {best_epoch+1}")

    # Save history
    with open(fold_dir / 'history.json', 'w') as f:
        json.dump(history, f, indent=2)

    # Clear model (keep volume cache for next fold)
    del model, optimizer, scheduler, scaler
    del train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

    return {
        'fold': fold,
        'best_lb_score': best_lb_score,
        'best_epoch': best_epoch,
        'history': history,
    }

print("Fold training ready (preload all - 220GB RAM)")

In [ ]:
# =============================================================================
# CELL 9: RUN 3-FOLD CV
# =============================================================================

def run_3fold_cv():
    """
    Run full 3-fold CV training.
    """
    if not FOLD_SPLITS:
        print("No fold splits available")
        return
    
    results = []
    
    for fold, (train_ids, val_ids) in enumerate(FOLD_SPLITS):
        gc.collect()
        torch.cuda.empty_cache()
        
        result = train_fold_v10(fold, train_ids, val_ids)
        results.append(result)
    
    print("\n" + "="*70)
    print("3-FOLD CV COMPLETE")
    print("="*70)
    for r in results:
        print(f"  Fold {r['fold']}: LB={r['best_lb_score']:.4f} @ epoch {r['best_epoch']+1}")
    print(f"  Mean LB: {np.mean([r['best_lb_score'] for r in results]):.4f}")
    print("="*70)
    
    return results

# Run training
# cv_results = run_3fold_cv()

print("Ready to run: cv_results = run_3fold_cv()")

In [ ]:
# =============================================================================
# CELL 10: OOF POSTPROCESS GRID SEARCH
# =============================================================================

def search_postprocess_params(
    oof_probs: Dict[str, np.ndarray],
    oof_gts: Dict[str, np.ndarray],
) -> Dict[str, float]:
    """
    Grid search for optimal postprocessing parameters.
    
    Parameters:
    - threshold: [0.20, 0.25, 0.30, 0.35, 0.40]
    - hysteresis: on/off with low/high combos
    - min_component_size: [0, 50, 100, 200]
    - closing_iters: [0, 1]
    
    Returns:
        Best parameters dict
    """
    best_score = -1
    best_params = {}
    
    thresholds = [0.20, 0.25, 0.30, 0.35, 0.40]
    hysteresis_configs = [
        None,  # off
        (0.16, 0.38),
        (0.15, 0.35),
        (0.18, 0.40),
    ]
    min_sizes = [0, 50, 100]
    closing_iters_list = [0, 1]
    
    print("Searching postprocess parameters...")
    
    for thresh in thresholds:
        for hyst in hysteresis_configs:
            for min_size in min_sizes:
                for closing in closing_iters_list:
                    scores = []
                    
                    for vid in oof_probs:
                        prob = oof_probs[vid]
                        gt = oof_gts[vid]
                        
                        # Apply threshold
                        if hyst is not None:
                            low, high = hyst
                            binary = np.zeros_like(prob, dtype=np.uint8)
                            binary[prob >= high] = 1
                            # Hysteresis: include low-threshold voxels connected to high
                            from scipy.ndimage import binary_dilation
                            seed = binary.copy()
                            mask = (prob >= low)
                            for _ in range(5):
                                seed = binary_dilation(seed) & mask
                            binary = seed.astype(np.uint8)
                        else:
                            binary = (prob >= thresh).astype(np.uint8)
                        
                        # Remove small components
                        if min_size > 0:
                            labeled = connected_components_3d(binary)
                            for i in range(1, labeled.max() + 1):
                                if (labeled == i).sum() < min_size:
                                    binary[labeled == i] = 0
                        
                        # Closing
                        if closing > 0:
                            struct = ndimage.generate_binary_structure(3, 1)
                            binary = ndimage.binary_closing(binary, structure=struct, iterations=closing)
                            binary = binary.astype(np.uint8)
                        
                        # Evaluate
                        metrics = evaluate_leaderboard(binary, gt)
                        scores.append(metrics['lb_score'])
                    
                    avg_score = np.mean(scores)
                    
                    if avg_score > best_score:
                        best_score = avg_score
                        best_params = {
                            'threshold': thresh,
                            'hysteresis': hyst,
                            'min_component_size': min_size,
                            'closing_iters': closing,
                            'lb_score': avg_score,
                        }
    
    print(f"\nBest params: {best_params}")
    return best_params

print("Postprocess search ready")

In [ ]:
# =============================================================================
# CELL 11: ENSEMBLE INFERENCE
# =============================================================================

def load_fold_models(checkpoint_dir: Path, n_folds: int = 3, device: str = 'cuda'):
    """
    Load best models from each fold.
    
    Returns:
        List of (model, score) tuples sorted by score descending
    """
    models = []
    
    for fold in range(n_folds):
        fold_path = checkpoint_dir / f"fold_{fold}" / "best_model.pth"
        if not fold_path.exists():
            print(f"Warning: Fold {fold} not found")
            continue
        
        ckpt = torch.load(fold_path, map_location=device, weights_only=False)
        
        model = ResEncUNet3D(
            features=cfg.FEATURES,
            n_blocks=cfg.N_BLOCKS,
            use_scse=cfg.USE_SCSE,
            use_deep_supervision=False,
        ).to(device)
        
        state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
        model.load_state_dict(state, strict=False)
        model.eval()
        
        score = ckpt.get('best_lb_score', 0)
        models.append((model, score, fold))
        print(f"Loaded fold {fold}: LB={score:.4f}")
    
    # Sort by score descending
    models.sort(key=lambda x: x[1], reverse=True)
    return models


@torch.no_grad()
def ensemble_predict(models_with_scores, volume, patch_size=(128,128,128), overlap=0.5, device='cuda'):
    """
    Ensemble prediction with score-weighted averaging.
    """
    # Normalize weights
    scores = np.array([s for _, s, _ in models_with_scores])
    weights = scores / scores.sum()
    
    ensemble_prob = None
    
    for (model, score, fold), weight in zip(models_with_scores, weights):
        prob = predict_volume(model, volume, patch_size, overlap, device)
        
        if ensemble_prob is None:
            ensemble_prob = prob * weight
        else:
            ensemble_prob += prob * weight
    
    return ensemble_prob


def run_submission(
    models_with_scores,
    test_ids: List[str],
    test_images_dir: Path,
    output_dir: Path,
    postprocess_params: Dict,
) -> Path:
    """
    Generate submission.zip
    """
    import zipfile
    import tifffile
    
    output_dir.mkdir(parents=True, exist_ok=True)
    
    for tid in tqdm(test_ids, desc="Predicting"):
        # Load volume
        vol = load_volume(test_images_dir / f"{tid}.tif")
        orig_shape = vol.shape
        orig_dtype = vol.dtype
        
        # Predict
        prob = ensemble_predict(models_with_scores, vol, cfg.PATCH_SIZE, overlap=0.5, device=cfg.DEVICE)
        
        # Postprocess
        thresh = postprocess_params.get('threshold', 0.30)
        hyst = postprocess_params.get('hysteresis', None)
        min_size = postprocess_params.get('min_component_size', 50)
        closing = postprocess_params.get('closing_iters', 0)
        
        if hyst is not None:
            low, high = hyst
            binary = np.zeros_like(prob, dtype=np.uint8)
            binary[prob >= high] = 1
            from scipy.ndimage import binary_dilation
            seed = binary.copy()
            mask = (prob >= low)
            for _ in range(5):
                seed = binary_dilation(seed) & mask
            binary = seed.astype(np.uint8)
        else:
            binary = (prob >= thresh).astype(np.uint8)
        
        if min_size > 0:
            labeled = connected_components_3d(binary)
            for i in range(1, labeled.max() + 1):
                if (labeled == i).sum() < min_size:
                    binary[labeled == i] = 0
        
        if closing > 0:
            struct = ndimage.generate_binary_structure(3, 1)
            binary = ndimage.binary_closing(binary, structure=struct, iterations=closing)
            binary = binary.astype(np.uint8)
        
        # Save
        tifffile.imwrite(output_dir / f"{tid}.tif", binary.astype(np.uint8))
    
    # Create zip
    zip_path = output_dir / "submission.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for tid in test_ids:
            zf.write(output_dir / f"{tid}.tif", f"{tid}.tif")
    
    print(f"Submission saved to {zip_path}")
    return zip_path

print("Ensemble inference ready")

In [ ]:
# =============================================================================
# CELL 12: SCHEDULE ACTIVATION TEST
# =============================================================================

def test_schedule_activation():
    """
    Test that topology losses actually activate within 300 epochs.
    """
    criterion = CombinedLossV10(
        skel_start=cfg.SKELETON_START_EPOCH,
        skel_warmup=cfg.SKELETON_WARMUP_EPOCHS,
        cldice_start=cfg.CLDICE_START_EPOCH,
        cldice_warmup=cfg.CLDICE_WARMUP_EPOCHS,
    )
    
    test_epochs = [0, 45, 90, 120, 180, 260, 299]
    
    print("Schedule activation test:")
    print(f"{'Epoch':<8} {'Skel Scale':<12} {'clDice Scale':<12} {'Phase'}")
    print("-" * 50)
    
    for epoch in test_epochs:
        skel_scale = criterion._scale(epoch, criterion.skel_start, criterion.skel_warmup)
        cldice_scale = criterion._scale(epoch, criterion.cldice_start, criterion.cldice_warmup)
        
        if epoch < cfg.SKELETON_START_EPOCH:
            phase = "DICE-ONLY"
        elif epoch < cfg.CLDICE_START_EPOCH:
            phase = "DICE+SKEL"
        else:
            phase = "FULL-TOPO"
        
        print(f"{epoch:<8} {skel_scale:<12.2f} {cldice_scale:<12.2f} {phase}")
    
    # Verify topology activates
    final_skel = criterion._scale(299, criterion.skel_start, criterion.skel_warmup)
    final_cldice = criterion._scale(299, criterion.cldice_start, criterion.cldice_warmup)
    
    assert final_skel == 1.0, f"Skeleton not fully active at epoch 299: {final_skel}"
    assert final_cldice == 1.0, f"clDice not fully active at epoch 299: {final_cldice}"
    
    print("\n✓ Schedule test PASSED - topology losses activate within 300 epochs")

test_schedule_activation()

In [ ]:
# =============================================================================
# CELL 13: 7-DAY RUNBOOK
# =============================================================================

print("""
╔══════════════════════════════════════════════════════════════════════╗
║                     7-DAY RUNBOOK                                     ║
╠══════════════════════════════════════════════════════════════════════╣
║ Day 1: Sanity check                                                   ║
║   - Run 20-epoch fold 0 to verify setup                               ║
║   - Check schedule activation (Cell 12)                               ║
║   - Verify no train/val overlap                                       ║
╠══════════════════════════════════════════════════════════════════════╣
║ Day 2-4: Full training                                                ║
║   - Run: cv_results = run_3fold_cv()                                  ║
║   - Monitor LB-aligned validation every 20 epochs                     ║
║   - Early stop if no improvement for 60 epochs                        ║
╠══════════════════════════════════════════════════════════════════════╣
║ Day 5: OOF calibration                                                ║
║   - Load OOF predictions from each fold                               ║
║   - Run: best_params = search_postprocess_params(oof_probs, oof_gts)  ║
║   - Calibrate ensemble weights                                        ║
╠══════════════════════════════════════════════════════════════════════╣
║ Day 6: Submit variants                                                ║
║   - Submit 2 controlled variants (postprocess A/B)                    ║
║   - Compare public LB movement                                        ║
║   - Select best config                                                ║
╠══════════════════════════════════════════════════════════════════════╣
║ Day 7: Final submission                                               ║
║   - Lock best config                                                  ║
║   - Clean training/inference run                                      ║
║   - Verify runtime < 9 hours                                          ║
╚══════════════════════════════════════════════════════════════════════╝

Target: 0.60+ LB | Stretch: 0.62-0.66
""")